Primero se define el problema a resolver:
    hallar la energia de una molecula H2

- Necesitamos hallar el Hamiltoniano del sistema: es un Hamiltoniano fermionico
los metodos vienen en qiskit nature package

In [1]:
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver

# definimos una molecula de dos atomos de hidrogeno localizados a una distancia de 0.735 A
# que es cercano al estado de equilibrio para esta molecula
driver = PySCFDriver(
    atom='H .0 .0 .0; H .0 .0 0.735',
    unit=DistanceUnit.ANGSTROM,
    basis='sto3g'
)

problem = driver.run()
secqop = problem.second_q_ops()
for item in secqop:
    print(item)


Fermionic Operator
number spin orbitals=4, number terms=36
  -1.25633907300325 * ( +_0 -_0 )
+ -0.47189600728114184 * ( +_1 -_1 )
+ -1.25633907300325 * ( +_2 -_2 )
+ -0.47189600728114184 * ( +_3 -_3 )
+ 0.33785507740175824 * ( +_0 +_0 -_0 -_0 )
+ 0.3322908651276482 * ( +_0 +_1 -_1 -_0 )
+ 0.33785507740175824 * ( +_0 +_2 -_2 -_0 )
+ 0.3322908651276482 * ( +_0 +_3 -_3 -_0 )
+ 0.09046559989211572 * ( +_0 +_0 -_1 -_1 )
+ 0.09046559989211572 * ( +_0 +_1 -_0 -_1 )
+ 0.09046559989211572 * ( +_0 +_2 -_3 -_1 )
+ 0.09046559989211572 * ( +_0 +_3 -_2 -_1 )
+ 0.09046559989211572 * ( +_1 +_0 -_1 -_0 )
+ 0.09046559989211572 * ( +_1 +_1 -_0 -_0 )
+ 0.09046559989211572 * ( +_1 +_2 -_3 -_0 )
+ 0.09046559989211572 * ( +_1 +_3 -_2 -_0 )
+ 0.3322908651276482 * ( +_1 +_0 -_0 -_1 )
+ 0.34928686136600906 * ( +_1 +_1 -_1 -_1 )
+ 0.3322908651276482 * ( +_1 +_2 -_2 -_1 )
+ 0.34928686136600906 * ( +_1 +_3 -_3 -_1 )
+ 0.33785507740175824 * ( +_2 +_0 -_0 -_2 )
+ 0.3322908651276482 * ( +_2 +_1 -_1 -_2 )
+ 0.33785507

In [5]:
from qiskit_nature.second_q.mappers import ParityMapper

mapper = ParityMapper(num_particles=problem.num_particles)

from qiskit_algorithms.optimizers import L_BFGS_B

optimizer = L_BFGS_B()

from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

second_q_op = problem.hamiltonian.second_q_op()
qubit_op = mapper.map(second_q_op)

from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

ansatz = UCCSD(
    num_spatial_orbitals=problem.num_spatial_orbitals, 
    num_particles=problem.num_particles,
    qubit_mapper=mapper,
    initial_state=HartreeFock(
        num_spatial_orbitals=problem.num_spatial_orbitals,
        num_particles=problem.num_particles,
        qubit_mapper=mapper
    )
)

ansatz.draw()

┌─────────────────────────────┐
q_0: ┤0                            ├
     │  EvolvedOps(t[0],t[1],t[2]) │
q_1: ┤1                            ├
     └─────────────────────────────┘

In [3]:
ansatz.decompose().draw()

┌───┐┌────────────────────┐┌────────────────────┐»
q_0: ┤ X ├┤0                   ├┤0                   ├»
     └───┘│  exp(-it IY)(t[0]) ││  exp(-it YI)(t[1]) │»
q_1: ─────┤1                   ├┤1                   ├»
          └────────────────────┘└────────────────────┘»
«     ┌───────────────────────────┐
«q_0: ┤0                          ├
«     │  exp(-it (XY + YX))(t[2]) │
«q_1: ┤1                          ├
«     └───────────────────────────┘

In [8]:
from qiskit_algorithms import VQE

vqe = VQE(ansatz=ansatz, optimizer=optimizer, estimator=estimator)
vqe.initial_point = [0.0]*ansatz.num_parameters
result = vqe.compute_minimum_eigenvalue(qubit_op)
print(f'eigenvalue: {result.eigenvalue}')


/opt/miniconda3/lib/python3.12/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:603: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/opt/miniconda3/lib/python3.12/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


eigenvalue: -1.857275030202378
